This notebook is intended for data analysis seperate from data aquisition and after database creation to make this project easier to use.

In [2]:
#necessary imports

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from scipy import stats

In [4]:
#Establish database connection
db_path = '../data/NPS_Mortality_Project.db'
#create connection object
conn = sqlite3.connect(db_path)

In [ ]:
#test connection
test_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(test_query, conn)

print(f"\n({len(tables)}):")
for table in tables['name']:
    print(f"  - {table}")


(12):
  - park_units
  - dates
  - cause_types
  - intent_types
  - sex_types
  - activity_types
  - amenity_types
  - mortality_incidents
  - amenities
  - park_activities
  - visits
  - geo


In [7]:
#Establish style guide
plt.style.use('seaborn-v0_8-darkgrid')

#Figure size(width, height in inches)
plt.rcParams['figure.figsize'] = (12, 8)

#font sizes
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

#colors
sns.set_palette("husl")

I'm going to start analysis by establishing the baseline mortality rate for all data in all parks. I'll combine all visitor counts and death counts to calculate a rate of fatal events per million visitors.

In [13]:
query1 = """
WITH park_visitors AS (
    SELECT 
        park_id,
        SUM(recreation_visitors + nonrecreation_visitors) as total_visitors
    FROM visits
    GROUP BY park_id
),
park_deaths AS (
    SELECT 
        park_id,
        COUNT(incident_id) as total_deaths
    FROM mortality_incidents
    GROUP BY park_id
)
SELECT 
    pu.park_name,
    pu.key_code,
    pu.region,
    COALESCE(pd.total_deaths, 0) as total_deaths,
    pv.total_visitors,
    CASE 
        WHEN pv.total_visitors > 0 THEN 
            ROUND((COALESCE(pd.total_deaths, 0) * 1000000.0) / pv.total_visitors, 2)
        ELSE 0
    END as mortality_rate
FROM park_units pu
INNER JOIN park_visitors pv ON pu.park_id = pv.park_id
LEFT JOIN park_deaths pd ON pu.park_id = pd.park_id
WHERE pv.total_visitors > 0
ORDER BY mortality_rate DESC;
"""

df_park_rates = pd.read_sql(query1, conn)

print("=" * 80)
print("PARK MORTALITY RATES (Deaths per Million Visitors, 2007-2024)")
print("=" * 80)
print(f"Total parks analyzed: {len(df_park_rates)}")
print(f"\nTop 20 Most Dangerous Parks:")
print("=" * 80)
display(df_park_rates.head(20))

print("\n" + "=" * 80)
print("SUMMARY STATISTICS:")
print("=" * 80)
print(f"Total deaths: {df_park_rates['total_deaths'].sum():,}")
print(f"Total visitors: {df_park_rates['total_visitors'].sum():,}")
overall_rate = (df_park_rates['total_deaths'].sum() / df_park_rates['total_visitors'].sum() * 1000000)
print(f"Overall NPS mortality rate: {overall_rate:.2f} per 1M visitors")
print(f"\nHighest mortality rate: {df_park_rates['mortality_rate'].max():.2f} per 1M visitors")
print(f"Parks with zero deaths: {len(df_park_rates[df_park_rates['total_deaths'] == 0]):,}")
print(f"Parks with deaths: {len(df_park_rates[df_park_rates['total_deaths'] > 0]):,}")

PARK MORTALITY RATES (Deaths per Million Visitors, 2007-2024)
Total parks analyzed: 397

Top 20 Most Dangerous Parks:


,park_name,key_code,region,total_deaths,total_visitors,mortality_rate
0,North Cascades National Park,noca,pwr,27,454221,59.44
1,Clara Barton National Historic Site,clba,ncr,8,157160,50.90
2,Saint Croix Island International Historic Site,sacr,ner,5,129691,38.55
3,Lake Clark National Park & Preserve,lacl,akr,5,244499,20.45
4,Fort Bowie National Historic Site,fobo,imr,2,148501,13.47
5,First State National Historical Park,frst,ner,2,225090,8.89
6,Katmai National Park & Preserve,katm,akr,6,744035,8.06
7,Gauley River National Recreation Area,gari,ner,16,2175049,7.36
8,Isle Royale National Park,isro,mwr,2,344464,5.81
9,Dry Tortugas National Park,drto,ser,7,1227306,5.70



SUMMARY STATISTICS:
Total deaths: 4,234
Total visitors: 8,036,820,549
Overall NPS mortality rate: 0.53 per 1M visitors

Highest mortality rate: 59.44 per 1M visitors
Parks with zero deaths: 215
Parks with deaths: 182


I am really surprised to see Clara Barton NHS on the top list. It won't really affect my analysis strategy, but I want to quickly see what the COD is for those.

In [19]:
check_clara = """
SELECT 
    ct.cause_name,
    COUNT(*) as deaths
FROM mortality_incidents mi
JOIN park_units pu ON mi.park_id = pu.park_id
JOIN cause_types ct ON mi.cause_id = ct.cause_id
WHERE pu.key_code = 'clba'
GROUP BY ct.cause_name
ORDER BY deaths DESC;
"""

df_clara_causes = pd.read_sql(check_clara, conn)
df_clara_causes

,cause_name,deaths
0,motor vehicle crash,7
1,medical - not during physical activity,1


In [18]:
clara_all = """
SELECT 
    mi.incident_date,
    d.year,
    d.month,
    d.day,
    ct.cause_name,
    ct.cause_category,
    it.intent_name,
    at.activity_name,
    mi.age_min,
    mi.age_max
FROM mortality_incidents AS mi
JOIN park_units      AS pu ON mi.park_id    = pu.park_id
LEFT JOIN dates      AS d  ON mi.date_id    = d.date_id
LEFT JOIN cause_types     AS ct ON mi.cause_id   = ct.cause_id
LEFT JOIN intent_types    AS it ON mi.intent_id  = it.intent_id
LEFT JOIN activity_types  AS at ON mi.activity_id = at.activity_id
WHERE pu.key_code = 'clba'
ORDER BY mi.incident_date;
"""

df_clara_all = pd.read_sql(clara_all, conn)
df_clara_all


,incident_date,year,month,day,cause_name,cause_category,intent_name,activity_name,age_min,age_max
0,2016-11-22 00:00:00,2016,11,22,motor vehicle crash,motor vehicle crash,unintentional,driving,55.0,64.0
1,2016-12-01 00:00:00,2016,12,1,motor vehicle crash,motor vehicle crash,unintentional,driving,15.0,24.0
2,2018-05-17 00:00:00,2018,5,17,medical - not during physical activity,medical - not during physical activity,medical,driving,65.0,NaN
3,2018-12-01 00:00:00,2018,12,1,motor vehicle crash,motor vehicle crash,unintentional,driving,15.0,24.0
4,2021-02-15 00:00:00,2021,2,15,motor vehicle crash,motor vehicle crash,unintentional,driving,45.0,54.0
5,2022-03-20 00:00:00,2022,3,20,motor vehicle crash,motor vehicle crash,unintentional,driving,15.0,24.0
6,2022-03-20 00:00:00,2022,3,20,motor vehicle crash,motor vehicle crash,unintentional,walking,65.0,NaN
7,2023-02-18 00:00:00,2023,2,18,motor vehicle crash,motor vehicle crash,unintentional,driving,65.0,NaN


Clara Barton NHS is accessible from the Clara Barton Parkway, a major thoroughfare in Maryland. I was also able to find a news story confirming at least one of these deaths was on the parkway. (https://wtop.com/montgomery-county/2023/02/crash-on-clara-barton-parkway-leaves-2-hospitalized/). I have high confidence that this data is not solely representative of the historic site itself. I do not consider it accurate to believe this site is more dangerous as the calculated rate made it seem. Further, the medical death occured while driving. There are no road systems at CBNHS, so I feel this entry is also erroneous.

I'm going to build a list of parks that should be excluded from analysis. I don't know if I'll find any others to add, but I'll start this way so I can build a function that will not have to be updated if additional data quality issues are found.

In [21]:
EXCLUDED_PARK_CODES = {"clba"}

In [20]:
def exclude_parks(df, code_col="key_code"):
    return df[~df[code_col].str.lower().isin(EXCLUDED_PARK_CODES)]


In [22]:
SQL_EXCLUDE = "pu.key_code NOT IN ('clba')"

In [23]:
query1 = f"""
WITH park_visitors AS (
    SELECT 
        park_id,
        SUM(recreation_visitors + nonrecreation_visitors) as total_visitors
    FROM visits
    GROUP BY park_id
),
park_deaths AS (
    SELECT 
        park_id,
        COUNT(incident_id) as total_deaths
    FROM mortality_incidents
    GROUP BY park_id
)
SELECT 
    pu.park_name,
    pu.key_code,
    pu.region,
    COALESCE(pd.total_deaths, 0) as total_deaths,
    pv.total_visitors,
    CASE 
        WHEN pv.total_visitors > 0 THEN 
            ROUND((COALESCE(pd.total_deaths, 0) * 1000000.0) / pv.total_visitors, 2)
        ELSE 0
    END as mortality_rate
FROM park_units pu
INNER JOIN park_visitors pv ON pu.park_id = pv.park_id
LEFT JOIN park_deaths pd ON pu.park_id = pd.park_id
WHERE pv.total_visitors > 0
  AND {SQL_EXCLUDE}       -- ⬅️ Your global exclusion goes here
ORDER BY mortality_rate DESC;
"""

df_park_rates = pd.read_sql(query1, conn)

print("=" * 80)
print("PARK MORTALITY RATES (Deaths per Million Visitors, 2007-2024)")
print("=" * 80)
print(f"Total parks analyzed: {len(df_park_rates)}")
print(f"\nTop 20 Most Dangerous Parks:")
print("=" * 80)
display(df_park_rates.head(20))

print("\n" + "=" * 80)
print("SUMMARY STATISTICS:")
print("=" * 80)
print(f"Total deaths: {df_park_rates['total_deaths'].sum():,}")
print(f"Total visitors: {df_park_rates['total_visitors'].sum():,}")
overall_rate = (df_park_rates['total_deaths'].sum() / df_park_rates['total_visitors'].sum() * 1000000)
print(f"Overall NPS mortality rate: {overall_rate:.2f} per 1M visitors")
print(f"\nHighest mortality rate: {df_park_rates['mortality_rate'].max():.2f} per 1M visitors")
print(f"Parks with zero deaths: {len(df_park_rates[df_park_rates['total_deaths'] == 0]):,}")
print(f"Parks with deaths: {len(df_park_rates[df_park_rates['total_deaths'] > 0]):,}")

PARK MORTALITY RATES (Deaths per Million Visitors, 2007-2024)
Total parks analyzed: 396

Top 20 Most Dangerous Parks:


,park_name,key_code,region,total_deaths,total_visitors,mortality_rate
0,North Cascades National Park,noca,pwr,27,454221,59.44
1,Saint Croix Island International Historic Site,sacr,ner,5,129691,38.55
2,Lake Clark National Park & Preserve,lacl,akr,5,244499,20.45
3,Fort Bowie National Historic Site,fobo,imr,2,148501,13.47
4,First State National Historical Park,frst,ner,2,225090,8.89
5,Katmai National Park & Preserve,katm,akr,6,744035,8.06
6,Gauley River National Recreation Area,gari,ner,16,2175049,7.36
7,Isle Royale National Park,isro,mwr,2,344464,5.81
8,Dry Tortugas National Park,drto,ser,7,1227306,5.70
9,Big Thicket National Preserve,bith,imr,16,3168060,5.05



SUMMARY STATISTICS:
Total deaths: 4,226
Total visitors: 8,036,663,389
Overall NPS mortality rate: 0.53 per 1M visitors

Highest mortality rate: 59.44 per 1M visitors
Parks with zero deaths: 215
Parks with deaths: 181


I want to look at the COVID timeframe because I know there were significant unusual changes in visitation during this period. I'm going to see what the average rates looked liek in pre- (2015-2019) during- (2020-2021) and post- (2022-) covid years.

In [25]:
query_period_rates = f"""
WITH visitors_period AS (
    SELECT
        v.park_id,
        CASE
            WHEN v.year BETWEEN 2015 AND 2019 THEN 'pre_covid_5yr'
            WHEN v.year BETWEEN 2020 AND 2021 THEN 'during_covid'
            WHEN v.year >= 2022 THEN 'post_covid'
        END AS period,
        SUM(v.recreation_visitors + v.nonrecreation_visitors) AS total_visitors
    FROM visits v
    WHERE v.year >= 2015   -- ensure consistent data quality
    GROUP BY v.park_id, period
),
deaths_period AS (
    SELECT
        mi.park_id,
        CASE
            WHEN d.year BETWEEN 2015 AND 2019 THEN 'pre_covid_5yr'
            WHEN d.year BETWEEN 2020 AND 2021 THEN 'during_covid'
            WHEN d.year >= 2022 THEN 'post_covid'
        END AS period,
        COUNT(mi.incident_id) AS total_deaths
    FROM mortality_incidents mi
    JOIN dates d ON mi.date_id = d.date_id
    WHERE d.year >= 2015
    GROUP BY mi.park_id, period
),
park_period_rates AS (
    SELECT
        pu.park_name,
        pu.key_code,
        vp.period,
        COALESCE(dp.total_deaths, 0) AS total_deaths,
        vp.total_visitors,
        CASE 
            WHEN vp.total_visitors > 0 THEN
                (COALESCE(dp.total_deaths, 0) * 1000000.0) / vp.total_visitors
            ELSE NULL
        END AS mortality_rate
    FROM park_units pu
    JOIN visitors_period vp ON pu.park_id = vp.park_id
    LEFT JOIN deaths_period dp
        ON vp.park_id = dp.park_id
       AND vp.period = dp.period
    WHERE {SQL_EXCLUDE}   -- your global exclusion (e.g., CLBA)
)
SELECT
    period,
    SUM(total_deaths) AS deaths_all_parks,
    SUM(total_visitors) AS visitors_all_parks,
    ROUND(
        SUM(total_deaths) * 1000000.0 
        / NULLIF(SUM(total_visitors), 0), 
        2
    ) AS overall_mortality_rate_per_million,
    ROUND(
        AVG(mortality_rate),
        2
    ) AS mean_park_mortality_rate_per_million,
    COUNT(DISTINCT key_code) AS parks_included
FROM park_period_rates
GROUP BY period
ORDER BY 
    CASE 
        WHEN period = 'pre_covid_5yr' THEN 1
        WHEN period = 'during_covid' THEN 2
        ELSE 3
    END;
"""

df_period_rates = pd.read_sql(query_period_rates, conn)
df_period_rates

,period,deaths_all_parks,visitors_all_parks,overall_mortality_rate_per_million,mean_park_mortality_rate_per_million,parks_included
0,pre_covid_5yr,1653,2472061231,0.67,1.05,378
1,during_covid,582,840690304,0.69,0.59,387
2,post_covid,648,1211232927,0.53,0.70,397


Looking only at 2015–2024, the overall NPS mortality rate per million visitors is relatively stable before and during COVID (0.67 vs 0.69 deaths per 1M visitors), with a notable decrease in the post-COVID period (0.53 deaths per 1M).

The average park-level rate drops sharply during COVID (from 1.05 to 0.59 deaths per 1M visitors), then increases somewhat post-COVID (0.70), while remaining below the pre-COVID baseline. It's possible that changes in visitation, activities, or some other demographic shift during COVID affected this rate. To further investigate, I want to look at the parks that had the greatest change in rate to see if there are any observable patterns.

In [ ]:
query_greatest_change = f"""
WITH visitors_period AS (
    SELECT
        v.park_id,
        CASE
            WHEN v.year BETWEEN 2015 AND 2019 THEN 'pre'
            WHEN v.year >= 2022 THEN 'post'
        END AS period,
        SUM(v.recreation_visitors + v.nonrecreation_visitors) AS total_visitors
    FROM visits v
    WHERE v.year BETWEEN 2015 AND 2024
    GROUP BY v.park_id, period
),
deaths_period AS (
    SELECT
        mi.park_id,
        CASE
            WHEN d.year BETWEEN 2015 AND 2019 THEN 'pre'
            WHEN d.year >= 2022 THEN 'post'
        END AS period,
        COUNT(mi.incident_id) AS total_deaths
    FROM mortality_incidents mi
    JOIN dates d ON mi.date_id = d.date_id
    WHERE d.year BETWEEN 2015 AND 2024
    GROUP BY mi.park_id, period
),
park_rates AS (
    SELECT
        pu.park_name,
        pu.key_code,
        vp.period,
        COALESCE(dp.total_deaths, 0) AS total_deaths,
        vp.total_visitors,
        CASE
            WHEN vp.total_visitors > 0 THEN
                (COALESCE(dp.total_deaths, 0) * 1000000.0) / vp.total_visitors
            ELSE NULL
        END AS mortality_rate
    FROM park_units pu
    JOIN visitors_period vp ON pu.park_id = vp.park_id
    LEFT JOIN deaths_period dp
           ON vp.park_id = dp.park_id
          AND vp.period = dp.period
    WHERE {SQL_EXCLUDE}   -- global exclusion
),
pivoted AS (
    SELECT
        park_name,
        key_code,
        MAX(CASE WHEN period = 'pre' THEN mortality_rate END) AS pre_rate,
        MAX(CASE WHEN period = 'post' THEN mortality_rate END) AS post_rate
    FROM park_rates
    GROUP BY park_name, key_code
),
final AS (
    SELECT
        park_name,
        key_code,
        pre_rate,
        post_rate,
        (post_rate - pre_rate) AS absolute_change,
        CASE 
            WHEN pre_rate > 0 THEN ROUND(((post_rate - pre_rate) / pre_rate) * 100, 2)
            ELSE NULL
        END AS percent_change
    FROM pivoted
    WHERE pre_rate IS NOT NULL AND post_rate IS NOT NULL
)
SELECT *
FROM final
ORDER BY ABS(absolute_change) DESC
LIMIT 50;
"""

df_greatest_change = pd.read_sql(query_greatest_change, conn)
df_greatest_change


,park_name,key_code,pre_rate,post_rate,absolute_change,percent_change
0,Fort Bowie National Historic Site,fobo,24.849660,0.000000,-24.849660,-100.00
1,Saint Croix Island International Historic Site,sacr,16.155089,0.000000,-16.155089,-100.00
2,Gauley River National Recreation Area,gari,1.795155,12.128357,10.333202,575.62
3,Niobrara National Scenic River,niob,8.691118,0.000000,-8.691118,-100.00
4,Katmai National Park & Preserve,katm,8.494769,0.000000,-8.494769,-100.00
5,North Cascades National Park,noca,60.834651,52.619117,-8.215535,-13.50
6,Isle Royale National Park,isro,8.009547,0.000000,-8.009547,-100.00
7,Dry Tortugas National Park,drto,10.939483,4.645955,-6.293528,-57.53
8,Organ Pipe Cactus National Monument,orpi,7.917807,2.456499,-5.461307,-68.98
9,Pecos National Historical Park,peco,5.166437,0.000000,-5.166437,-100.00
